In [14]:
# 1. Wipe the corrupted pre-installed versions
!pip uninstall -y transformers tokenizers
# 2. Install a clean, synchronized stack
!pip install -q -U \
    transformers \
    tokenizers \
    peft \
    trl==0.12.1 \
    accelerate \
    bitsandbytes \
    datasets \
    einops \
    huggingface_hub

print("✅ Clean dependencies installed")

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.5 MB/s eta 0:00:00a 0:00:01
✅ Clean dependencies installed


In [15]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

Cell 2: Verify GPU (unchanged)

In [16]:
import torch
import bitsandbytes as bnb

assert torch.cuda.is_available(), "❌ GPU not available — enable GPU in Kaggle settings"
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"✅ bnb version: {bnb.__version__}")
print(f"✅ PyTorch version: {torch.__version__}")

✅ CUDA available: True
✅ GPU: Tesla T4
✅ GPU memory: 15.6 GB
✅ bnb version: 0.49.2
✅ PyTorch version: 2.10.0+cu128


Cell 3: Configuration

In [17]:
import os
from pathlib import Path

# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — UPDATE THESE VALUES FOR YOUR RUN
# ════════════════════════════════════════════════════════════════════════════

# ── Dataset Path ────────────────────────────────────────────────────────────
# Kaggle mounts datasets under /kaggle/input/<dataset-name>/
# 
# BEFORE RUNNING:
# 1. Upload sft_train_final.jsonl + sft_val_final.jsonl as a Kaggle dataset
# 2. Add the dataset to this notebook (right sidebar → Add Input → Your dataset)
# 3. Update DATASET_PATH below to match YOUR dataset's mounted path
#    (Check the right sidebar after adding — it shows the exact path)
#
# Example: If your dataset is named "de-swarm-sft-v2", the path is:
#   /kaggle/input/de-swarm-sft-v2/
#
DATASET_PATH = "/kaggle/input/datasets/nurahmaddata/de-swarm-v2"  # ← UPDATE THIS

TRAIN_FILE = os.path.join(DATASET_PATH, "sft_train_final.jsonl")
VAL_FILE = os.path.join(DATASET_PATH, "sft_val_final.jsonl")

# ── Output Paths ────────────────────────────────────────────────────────────
# /kaggle/working/ is the ONLY directory that gets saved with the notebook
OUTPUT_DIR = Path("/kaggle/working/sft_output")
MERGED_MODEL_DIR = Path("/kaggle/working/qwen-coder-3b-sql-sft-merged")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ───────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

# v2 CHANGE: Increased from 1024 → 2048
# Reason: SaaS schema context is ~10K chars (~2500 tokens). With the prompt +
# SQL output, total can reach 1500+ tokens. 2048 gives safe headroom.
MAX_SEQ_LENGTH = 1024

# ── QLoRA Configuration ─────────────────────────────────────────────────────
LORA_R = 16              # LoRA rank — controls adapter capacity
LORA_ALPHA = 32          # LoRA scaling factor (alpha = 2 × r is standard)
LORA_DROPOUT = 0.05      # Light dropout to prevent overfitting
LORA_TARGET_MODULES = [  # Target all linear layers for best SQL quality
    "q_proj", "k_proj", "v_proj", "o_proj",    # Attention layers
    "gate_proj", "up_proj", "down_proj",        # MLP layers
]

# ── Training Hyperparameters ────────────────────────────────────────────────
# v2 CHANGE: Reduced epochs from 3 → 2
# Reason: With 5,082 rows (12.8× more data than v1's 418), 2 epochs is
# sufficient. 3 epochs risks overfitting on the larger dataset.
NUM_EPOCHS = 2

LEARNING_RATE = 2e-4     # Standard for QLoRA
PER_DEVICE_BATCH = 4     # T4 can handle 4 with seq_len=2048
GRAD_ACCUM_STEPS = 4     # Effective batch = 4 × 4 = 16
WARMUP_RATIO = 0.05      # 5% of steps for warmup
WEIGHT_DECAY = 0.01      # Light regularization
LOGGING_STEPS = 10       # Log every 10 steps
SAVE_STEPS = 50          # Save checkpoint every 50 steps

USE_FP16 = True           # T4 supports fp16

# ── Sanity Checks ───────────────────────────────────────────────────────────
assert os.path.exists(TRAIN_FILE), f"❌ Train file not found: {TRAIN_FILE}"
assert os.path.exists(VAL_FILE), f"❌ Val file not found: {VAL_FILE}"

print("═" * 60)
print("  de-swarm v2 Training Configuration")
print("═" * 60)
print(f"  Train file      : {TRAIN_FILE}")
print(f"  Val file        : {VAL_FILE}")
print(f"  Model           : {MODEL_NAME}")
print(f"  Max seq length  : {MAX_SEQ_LENGTH}")
print(f"  LoRA config     : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  Epochs          : {NUM_EPOCHS}")
print(f"  Batch size      : {PER_DEVICE_BATCH} × {GRAD_ACCUM_STEPS} = {PER_DEVICE_BATCH * GRAD_ACCUM_STEPS}")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Output dir      : {MERGED_MODEL_DIR}")
print("═" * 60)

════════════════════════════════════════════════════════════
  de-swarm v2 Training Configuration
════════════════════════════════════════════════════════════
  Train file      : /kaggle/input/datasets/nurahmaddata/de-swarm-v2/sft_train_final.jsonl
  Val file        : /kaggle/input/datasets/nurahmaddata/de-swarm-v2/sft_val_final.jsonl
  Model           : Qwen/Qwen2.5-Coder-3B-Instruct
  Max seq length  : 1024
  LoRA config     : r=16, alpha=32, dropout=0.05
  Epochs          : 2
  Batch size      : 4 × 4 = 16
  Learning rate   : 0.0002
  Output dir      : /kaggle/working/qwen-coder-3b-sql-sft-merged
════════════════════════════════════════════════════════════


Cell 4: Load + inspect dataset

In [18]:
import json
from datasets import Dataset

def load_jsonl(path: str) -> list[dict]:
    """Load JSONL file and return list of dicts."""
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_FILE)
val_rows = load_jsonl(VAL_FILE)

print(f"Train rows : {len(train_rows)}")
print(f"Val rows   : {len(val_rows)}")
print()

# Show a sample row from each schema
print("=== Sample row ===")
print(json.dumps(train_rows[0], indent=2, ensure_ascii=False)[:600])

# Convert to HF datasets
train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

# Validate structure
assert "messages" in train_ds[0], "❌ Dataset missing 'messages' field"
assert len(train_ds[0]["messages"]) == 3, "❌ Each row should have 3 messages (system, user, assistant)"
print("\n✅ Dataset structure valid")

# Show token length distribution
import statistics
token_lengths = []
for row in train_rows[:500]:  # sample first 500
    text = " ".join(m["content"] for m in row["messages"])
    # Rough estimate: ~4 chars per token
    token_lengths.append(len(text) // 4)

print(f"\nToken length stats (first 500 rows, estimated):")
print(f"  Min    : {min(token_lengths)}")
print(f"  Max    : {max(token_lengths)}")
print(f"  Mean   : {statistics.mean(token_lengths):.0f}")
print(f"  Median : {statistics.median(token_lengths):.0f}")
print(f"  P95    : {sorted(token_lengths)[int(len(token_lengths) * 0.95)]}")

Train rows : 5082
Val rows   : 267

=== Sample row ===
{
  "messages": [
    {
      "role": "system",
      "content": "You are a SQL generation assistant. Given a natural language request and a database schema, output a single read-only SQL query that answers the request. Rules: output ONLY the SQL query ending with a semicolon. Use only tables and columns from the schema. Forbidden: DROP, DELETE, UPDATE, INSERT, ALTER, TRUNCATE, SELECT *. Use ANSI SQL syntax compatible with SQLite."
    },
    {
      "role": "user",
      "content": "What is the user base for Pro plan?"
    },
    {
      "role": "assistant",
      "content": "SELECT COUNT(DIS

✅ Dataset structure valid

Token length stats (first 500 rows, estimated):
  Min    : 115
  Max    : 372
  Mean   : 174
  Median : 167
  P95    : 243


Cell 5: Load tokenizer 

In [19]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"  # Right for training, left for inference

print(f"✅ Tokenizer loaded")
print(f"  vocab_size      : {len(tokenizer)}")
print(f"  pad_token       : {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"  eos_token       : {tokenizer.eos_token} (id={tokenizer.eos_token_id})")
print(f"  chat_template   : {'present' if tokenizer.chat_template else 'MISSING'}")

# Test the chat template on a sample
sample = train_rows[0]
rendered = tokenizer.apply_chat_template(
    sample["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print("\n--- Rendered sample (first 500 chars) ---")
print(rendered[:500])

✅ Tokenizer loaded
  vocab_size      : 151665
  pad_token       : <|endoftext|> (id=151643)
  eos_token       : <|im_end|> (id=151645)
  chat_template   : present

--- Rendered sample (first 500 chars) ---
<|im_start|>system
You are a SQL generation assistant. Given a natural language request and a database schema, output a single read-only SQL query that answers the request. Rules: output ONLY the SQL query ending with a semicolon. Use only tables and columns from the schema. Forbidden: DROP, DELETE, UPDATE, INSERT, ALTER, TRUNCATE, SELECT *. Use ANSI SQL syntax compatible with SQLite.<|im_end|>
<|im_start|>user
What is the user base for Pro plan?<|im_end|>
<|im_start|>assistant
SELECT COUNT(DIST


Cell 6: Load model in 4-bit 

In [20]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"✅ Model loaded in 4-bit")
print(f"  model dtype      : {next(model.parameters()).dtype}")
print(f"  trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"  total params     : {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model loaded in 4-bit
  model dtype      : torch.bfloat16
  trainable params : 311,406,592
  total params     : 1,698,672,640


Apply LoRA adapters 

In [21]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Cell 8: Configure traine

In [22]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    report_to="none",
    seed=42,
)

# Completion-only loss — mask system+user tokens, train on assistant response only
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

# Calculate expected steps
total_steps = (len(train_ds) * NUM_EPOCHS) // (PER_DEVICE_BATCH * GRAD_ACCUM_STEPS)

print("✅ Training arguments configured")
print(f"  effective batch size : {PER_DEVICE_BATCH * GRAD_ACCUM_STEPS}")
print(f"  total steps (est)    : ~{total_steps}")
print(f"  epochs               : {NUM_EPOCHS}")
print(f"  mixed precision      : {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")
print(f"  max seq length       : {MAX_SEQ_LENGTH}")
print(f"  eval every           : {SAVE_STEPS} steps")
print(f"  expected wall time   : ~{total_steps * 3 // 60} minutes on T4")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Training arguments configured
  effective batch size : 16
  total steps (est)    : ~635
  epochs               : 2
  mixed precision      : bf16
  max seq length       : 1024
  eval every           : 50 steps
  expected wall time   : ~31 minutes on T4


In [23]:
# 1. Define a function to flatten the messages using the tokenizer's built-in chat template
def apply_chat_template(row):
    row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False)
    return row

# 2. Apply it to both datasets (num_proc=2 utilizes your Kaggle CPU cores for speed)
print("Formatting training dataset...")
train_ds = train_ds.map(apply_chat_template, num_proc=2)

print("Formatting validation dataset...")
val_ds = val_ds.map(apply_chat_template, num_proc=2)

Formatting training dataset...


Map (num_proc=2):   0%|          | 0/5082 [00:00<?, ? examples/s]

Formatting validation dataset...


Map (num_proc=2):   0%|          | 0/267 [00:00<?, ? examples/s]

In [24]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
)
print("✅ Trainer initialized")

Map:   0%|          | 0/5082 [00:00<?, ? examples/s]

Map:   0%|          | 0/267 [00:00<?, ? examples/s]

✅ Trainer initialized


Train

In [25]:
import time

print("═" * 60)
print("  Starting training...")
print(f"  Dataset: {len(train_ds)} train rows, {len(val_ds)} val rows")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Expected time: ~{total_steps * 3 // 60} minutes on T4")
print("═" * 60)
print()

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f"\n✅ Training complete in {elapsed/60:.1f} minutes")
print(f"   That's {elapsed/len(train_ds):.2f}s per sample")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


════════════════════════════════════════════════════════════
  Starting training...
  Dataset: 5082 train rows, 267 val rows
  Epochs: 2
  Expected time: ~31 minutes on T4
════════════════════════════════════════════════════════════



Step,Training Loss,Validation Loss
50,0.244296,0.230359
100,0.172241,0.180201
150,0.172645,0.162784
200,0.119276,0.147900
250,0.133558,0.139327
300,0.114286,0.134212
350,0.098235,0.132305
400,0.093019,0.128921
450,0.088714,0.123756
500,0.080751,0.121691



✅ Training complete in 100.0 minutes
   That's 1.18s per sample


In [26]:
import json

log_history = trainer.state.log_history
print(f"Total log entries: {len(log_history)}\n")
print(f"{'Step':>5}  {'Train Loss':>12}  {'Val Loss':>12}  {'LR':>10}")
print("-" * 50)
for entry in log_history:
    step = entry.get("step", "?")
    if "loss" in entry:
        print(f"{step:>5}  {entry['loss']:>12.6f}  {'—':>12}  {entry.get('learning_rate', 0):.2e}")
    elif "eval_loss" in entry:
        print(f"{step:>5}  {'—':>12}  {entry['eval_loss']:>12.6f}  {'—':>10}")

final_train = [e for e in log_history if "loss" in e]
final_val = [e for e in log_history if "eval_loss" in e]
if final_train:
    print(f"\n📊 Final training loss: {final_train[-1]['loss']:.6f}")
if final_val:
    print(f"📊 Final validation loss: {final_val[-1]['eval_loss']:.6f}")
if final_train and final_val:
    gap = final_val[-1]['eval_loss'] - final_train[-1]['loss']
    print(f"📊 Train/Val gap: {gap:+.4f} ({'overfitting' if gap > 0.3 else 'healthy'})")

Total log entries: 77

 Step    Train Loss      Val Loss          LR
--------------------------------------------------
   10      0.667517             —  5.63e-05
   20      0.460887             —  1.19e-04
   30      0.335685             —  1.81e-04
   40      0.289842             —  2.00e-04
   50      0.244296             —  2.00e-04
   50             —      0.230359           —
   60      0.198070             —  1.99e-04
   70      0.185463             —  1.98e-04
   80      0.201159             —  1.97e-04
   90      0.174965             —  1.96e-04
  100      0.172241             —  1.94e-04
  100             —      0.180201           —
  110      0.141756             —  1.92e-04
  120      0.182298             —  1.90e-04
  130      0.152836             —  1.88e-04
  140      0.158048             —  1.85e-04
  150      0.172645             —  1.82e-04
  150             —      0.162784           —
  160      0.142465             —  1.79e-04
  170      0.153810             —  1.7

In [27]:
adapter_dir = OUTPUT_DIR / "final_adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

print(f"✅ Adapter saved → {adapter_dir}")
!ls -lh {adapter_dir}

✅ Adapter saved → /kaggle/working/sft_output/final_adapter
total 126M
-rw-r--r-- 1 root root 1.1K Jun 22 14:38 adapter_config.json
-rw-r--r-- 1 root root 115M Jun 22 14:38 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K Jun 22 14:38 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Jun 22 14:38 README.md
-rw-r--r-- 1 root root  693 Jun 22 14:38 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun 22 14:38 tokenizer.json
-rw-r--r-- 1 root root 5.4K Jun 22 14:38 training_args.bin


In [28]:
import gc
import subprocess
import sys

# ── 1. Fix the torchao version conflict ────────────────────────────────────
print("Installing compatible torchao version...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchao>=0.16.0"])

# ── 2. Clear memory (safely) ───────────────────────────────────────────────
try:
    del model
    del trainer
except NameError:
    pass  # Variables already cleared or kernel restarted

gc.collect()
torch.cuda.empty_cache()

# ── 3. Reload base model in fp16 ───────────────────────────────────────────
print("Reloading base model in fp16 for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# ── 4. Merge LoRA weights ──────────────────────────────────────────────────
from peft import PeftModel
merged = PeftModel.from_pretrained(base_model, str(adapter_dir))
merged = merged.merge_and_unload()

print("✅ LoRA weights merged successfully!")

Installing compatible torchao version...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.6 MB/s eta 0:00:00


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Reloading base model in fp16 for merging...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


✅ LoRA weights merged successfully!


In [29]:
merged.save_pretrained(str(MERGED_MODEL_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_MODEL_DIR))

print(f"✅ Merged model saved → {MERGED_MODEL_DIR}")
!ls -lh {MERGED_MODEL_DIR}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model saved → /kaggle/working/qwen-coder-3b-sql-sft-merged
total 5.8G
-rw-r--r-- 1 root root 2.5K Jun 22 14:38 chat_template.jinja
-rw-r--r-- 1 root root 1.6K Jun 22 14:38 config.json
-rw-r--r-- 1 root root  243 Jun 22 14:38 generation_config.json
-rw-r--r-- 1 root root 5.8G Jun 22 14:38 model.safetensors
-rw-r--r-- 1 root root  693 Jun 22 14:38 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun 22 14:38 tokenizer.json


In [30]:
import torch

merged.eval()

# v2 CHANGE: Test prompts from ALL 3 schemas
test_prompts = [
    # E-commerce schema
    "Show total revenue by region for the last 30 days",
    # SaaS schema
    "Show total organizations by plan",
    "Show MRR by plan for the last 30 days",
    # Retail star schema
    "Show total sales revenue by category for the last 30 days",
    "Show top 10 products by revenue for the last 60 days",
]

# Use the system prompt from training data
SYSTEM_PROMPT = train_rows[0]["messages"][0]["content"]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(merged.device)
    
    with torch.no_grad():
        outputs = merged.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    sql = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    print(f"\n--- PROMPT: {prompt}")
    print(f"--- SQL   : {sql}")
    print()


--- PROMPT: Show total revenue by region for the last 30 days
--- SQL   : SELECT region, SUM(total_amount) AS total_revenue FROM orders WHERE order_date >= DATE('now', '-30 day') GROUP BY region;


--- PROMPT: Show total organizations by plan
--- SQL   : SELECT plans.plan_name, COUNT(DISTINCT organizations.org_id) AS total_organizations FROM organizations JOIN subscriptions ON organizations.org_id = subscriptions.org_id JOIN plans ON subscriptions.plan_id = plans.plan_id WHERE subscriptions.status = 'active' GROUP BY plans.plan_name ORDER BY plans.plan_name ASC;


--- PROMPT: Show MRR by plan for the last 30 days
--- SQL   : SELECT p.plan_name, SUM(i.amount_paid) AS mrr FROM invoices i JOIN subscriptions s ON i.sub_id = s.sub_id JOIN plans p ON s.plan_id = p.plan_id WHERE i.status = 'paid' AND i.billing_period_end >= DATE('now', '-30 days') GROUP BY p.plan_name ORDER BY mrr DESC;


--- PROMPT: Show total sales revenue by category for the last 30 days
--- SQL   : SELECT dim_product.cat

In [31]:
import shutil
from pathlib import Path

source = Path("/kaggle/working/qwen-coder-3b-sql-sft-merged")
output_zip = Path("/kaggle/working/qwen-coder-3b-sql-sft-merged.zip")

if output_zip.exists():
    output_zip.unlink()

shutil.make_archive(
    base_name=str(output_zip.with_suffix("")),
    format="zip",
    root_dir=str(source.parent),
    base_dir=source.name,
)

size_gb = output_zip.stat().st_size / 1e9
print(f"✅ Created: {output_zip}")
print(f"   Size: {size_gb:.2f} GB")
print()
print("To download:")
print("1. Click 'Save Version' (top right)")
print("2. Choose 'Save' (NOT 'Save & Run All')")
print("3. After save, use: kaggle kernels output nurahmaddata/<notebook-slug> -p /path/to/dest")

✅ Created: /kaggle/working/qwen-coder-3b-sql-sft-merged.zip
   Size: 5.61 GB

To download:
1. Click 'Save Version' (top right)
2. Choose 'Save' (NOT 'Save & Run All')
3. After save, use: kaggle kernels output nurahmaddata/<notebook-slug> -p /path/to/dest


In [32]:
import os

# Check if the folder exists in the current live session
path = "/kaggle/working/qwen-coder-3b-sql-sft-merged"
if os.path.exists(path):
    files = os.listdir(path)
    print(f"✅ FOLDER EXISTS! It contains {len(files)} files:")
    for f in files:
        size = os.path.getsize(os.path.join(path, f)) / 1e6
        print(f"  - {f} ({size:.1f} MB)")
else:
    print("❌ Folder does NOT exist. The kernel was reset and the files are gone.")


✅ FOLDER EXISTS! It contains 6 files:
  - model.safetensors (6171.9 MB)
  - tokenizer_config.json (0.0 MB)
  - config.json (0.0 MB)
  - chat_template.jinja (0.0 MB)
  - tokenizer.json (11.4 MB)
  - generation_config.json (0.0 MB)


In [ ]:
# Install/Upgrade huggingface_hub
!pip install -q huggingface_hub

import os
from huggingface_hub import HfApi

# Retrieve your token from Kaggle secrets
hf_token = "YOUR_HUGGINGFACE_TOKEN_HERE" # Paste your token here if you didn't use Kaggle Secrets

# Initialize the API
api = HfApi(token=hf_token)

# Create a new model repository on HuggingFace
repo_id = "nurahmad-data/qwen-coder-3b-sql-v2" # Replace with your actual HF username
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Upload the entire merged model folder
api.upload_folder(
    folder_path="/kaggle/working/qwen-coder-3b-sql-sft-merged",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload fine-tuned Qwen 3B SQL model (v2)"
)

print(f"✅ Model uploaded to: https://huggingface.co/{repo_id}")

✅ Model uploaded to: https://huggingface.co/nurahmad-data/qwen-coder-3b-sql-v2
